# Week 2: Sentinel-2 Data Acquisition
### Lagos Barrier Island Shoreline Change Detection Project

**Purpose:** Query Google Earth Engine for the lowest-cloud-cover Sentinel-2 L2A scene per year (2017–2026) over the Lagos Barrier Island AOI, within the November–March dry-season window.

This notebook covers acquisition only. Cloud masking and the water index are handled in `02_preprocessing.ipynb`; thresholding and shoreline extraction are handled in `03_shoreline_extraction.ipynb`. Shared functions live in `pipeline_utils.py`, imported below.

**Note on study period:** the dataset starts at 2017, not 2016 — Sentinel-2 L2A coverage is not available for this AOI before March 2017 (confirmed by direct query, see README "Data Limitations").

## 1. Authenticate & Initialize Earth Engine

First run opens a browser window for Google sign-in. The explicit `scopes` list includes Drive access, needed by later stages that export data.

In [ ]:
import ee

ee.Authenticate(
    scopes=[
        "https://www.googleapis.com/auth/earthengine",
        "https://www.googleapis.com/auth/devstorage.full_control",
        "https://www.googleapis.com/auth/drive",
    ],
)
ee.Initialize()


## 2. Imports & Area of Interest (AOI)

In [ ]:
from pipeline_utils import get_best_scene
import geemap

aoi = ee.Geometry.Rectangle([3.30, 6.38, 3.55, 6.48])


## 3. Select the best scene per year (2017–2026)

In [ ]:
years = list(range(2017, 2026))  # dry-season windows: 2017-18 ... 2025-26

selected_scenes = {}
for yr in years:
    scene = get_best_scene(yr, aoi)
    if scene is not None:
        selected_scenes[yr] = scene

print(f"\nCompleted scene selection: {len(selected_scenes)} / {len(years)} years have a usable scene.")


## 4. Visual QA

In [ ]:
Map = geemap.Map(center=[6.43, 3.42], zoom=11)
for yr, img in selected_scenes.items():
    vis_params = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}
    Map.addLayer(img, vis_params, f"S2 {yr}")
Map.addLayer(aoi, {}, "AOI", opacity=0.3)
Map
